# Quora Question Pairs: Reproduce Results
This notebook:
1. Loads the pre-trained models stored in `models/`
2. Loads the fixed train/test/val splits
3. Computes predictions and metrics (ROC AUC, Precision, Recall)
4. Displays plots

No training happens here. Run `train_models.ipynb` first if the models are not stored yet.

In [22]:
from pathlib import Path
import pickle
import pandas as pd
import numpy as np
import importlib
import utils
importlib.reload(utils)

<module 'utils' from '/home/ntorquet/Documents/llm/TorquetLunaNuria/utils.py'>

## 1. Load the splits

In [23]:
train_df = pd.read_csv("data/splits/train_data.csv")
val_df = pd.read_csv("data/splits/val_data.csv")
test_df = pd.read_csv("data/splits/test_data.csv")

print("train_df.shape=", train_df.shape)
print("val_df.shape=",   val_df.shape)
print("test_df.shape=",  test_df.shape)

train_df.shape= (291897, 6)
val_df.shape= (15363, 6)
test_df.shape= (16172, 6)


## 2. Load models

In [24]:
MODEL_DIR = Path("models")

with open(MODEL_DIR / "count_vectorizer.pkl", "rb") as f:
    count_vectorizer = pickle.load(f)

with open(MODEL_DIR / "simple_model.pkl", "rb") as f:
    simple_model = pickle.load(f)

with open(MODEL_DIR / "enhanced_model.pkl", "rb") as f:
    enhanced_model = pickle.load(f)

with open(MODEL_DIR / "graph_features.pkl", "rb") as f:
    graph_features = pickle.load(f)
    q_freq = graph_features["q_freq"]
    pair_freq = graph_features["pair_freq"]

print("Models loaded")
print(" simple_model: ", type(simple_model).__name__)
print(" enhanced_model: ", type(enhanced_model).__name__)
print(" graph_features: ", len(q_freq), " unique questions")

Models loaded
 simple_model:  LogisticRegression
 enhanced_model:  GradientBoostingClassifier
 graph_features:  413822  unique questions


## 3. Build feature matrices
- Simple model uses BoW
- Enhanced model uses 16 features (statistical + nlp + graph)

In [25]:
print("Computing BoW features")
X_train_simple = utils.get_features_from_df(train_df, count_vectorizer)
X_val_simple = utils.get_features_from_df(val_df, count_vectorizer)
X_test_simple = utils.get_features_from_df(test_df, count_vectorizer)
print("BoW features ready", X_train_simple.shape)

Computing BoW features
BoW features ready (291897, 149650)


In [26]:
print("Computing enhanced features")
X_train_enhanced = utils.get_features(train_df, q_freq, pair_freq)
X_val_enhanced = utils.get_features(val_df, q_freq, pair_freq)
X_test_enhanced = utils.get_features(test_df, q_freq, pair_freq)
print("Enhanced features ready", X_train_enhanced.shape)

Computing enhanced features
Enhanced features ready (291897, 16)


## 4. Evaluate models on splits

In [27]:
y_train = train_df["is_duplicate"].values
y_val = val_df["is_duplicate"].values
y_test = test_df["is_duplicate"].values

results = []

for split_name, X, y in [
    ("train", X_train_simple, y_train),
    ("val", X_val_simple, y_val),
    ("test", X_test_simple, y_test)
]:
    row = utils.evaluate_model(simple_model, X, y, split_name)
    row["model"] = "simple_model"
    results.append(row)

for split_name, X, y in [
    ("train", X_train_enhanced, y_train),
    ("val", X_val_enhanced, y_val),
    ("test", X_test_enhanced, y_test)
]:
    row = utils.evaluate_model(enhanced_model, X, y, split_name)
    row["model"] = "enhanced_model"
    results.append(row)

## 5. Result summary

In [28]:
results_df = pd.DataFrame(results)[["model", "split", "accuracy", "roc_auc", "precision", "recall"]]
results_df = results_df.sort_values(by=["model", "split"]).reset_index(drop=True)
results_df

,model,split,accuracy,roc_auc,precision,recall
0,enhanced_model,test,0.8038,0.8935,0.8375,0.5853
1,enhanced_model,train,0.8319,0.9160,0.8117,0.7084
2,enhanced_model,val,0.8015,0.8912,0.8293,0.5814
3,simple_model,test,0.7578,0.8137,0.6955,0.6188
4,simple_model,train,0.8140,0.8899,0.7820,0.6867
5,simple_model,val,0.7491,0.8046,0.6773,0.6107


## 6. Feature importance (enhanced)

In [19]:
importance_df = pd.DataFrame({
    "feature": utils.FEATURE_NAMES,
    "importance": enhanced_model.feature_importances_
}).sort_values(by="importance", ascending=False).reset_index(drop=True)

importance_df

,feature,importance
0,freq_min,0.540401
1,char_trigram_similarity,0.137040
2,jaccard_no_stopwords,0.099901
3,shared_word_ratio_no_stopwords,0.054149
4,lcs_ratio_no_stopwords,0.038907
5,lcs_ratio,0.037145
6,length_diff_ratio,0.022742
7,tfidf_cosine_no_stopwords,0.016463
8,freq_diff,0.013648
9,freq_sum,0.012338


## 7. Error on test set

In [20]:
mistake_idx, preds = utils.get_mistakes(simple_model, X_test_simple, y_test)
print(f"Simple model: {len(mistake_idx)} / {len(y_test)} ({len(mistake_idx)/len(y_test):.1%})")

print("Examples of mistakes:")
utils.print_mistake_k(test_df.reset_index(drop=True), 0, mistake_idx, preds)

Simple model: 3917 / 16172 (24.2%)
Examples of mistakes:
Why can't I see views on my Instagram videos?
How can I see who views my Instagram?
True label: 1
Predicted label: 0


In [21]:
mistake_idx, preds = utils.get_mistakes(enhanced_model, X_test_enhanced, y_test)
print(f"Enhanced model: {len(mistake_idx)} / {len(y_test)} ({len(mistake_idx)/len(y_test):.1%})")

print("Examples of mistakes:")
utils.print_mistake_k(test_df.reset_index(drop=True), 0, mistake_idx, preds)

Enhanced model: 3173 / 16172 (19.6%)
Examples of mistakes:
Where is the best blood cancer hospital?
Where is the best blood cancer hospital in the world?
True label: 0
Predicted label: 1
